# 02 — Preprocessing & DuckDB
Clean data & establish database.

In [3]:
import pandas as pd
import numpy as np
import duckdb
from pathlib import Path

ROOT = Path().resolve().parents[0]
processed_dir = ROOT / 'data/processed'

df = pd.read_parquet(processed_dir / 'merged_hourly_raw.parquet')
print("Loaded raw shape:", df.shape)

Loaded raw shape: (34937, 11)


In [4]:
# handle missing values using ffill and city median
df['Start'] = pd.to_datetime(df['Start'])
df = df.sort_values(['City', 'Start'])

# fill gaps up to 3 hours with ffill
pollutants = ['PM2.5', 'PM10', 'NO2', 'O3']
for p in pollutants:
    if p in df.columns:
        df[p] = df.groupby('City')[p].ffill(limit=3)
        # fill remaining with median
        df[p] = df[p].fillna(df.groupby('City')[p].transform('median'))

print("Missing values handled.")

Missing values handled.


In [5]:
# duckdb setup
db_path = processed_dir / 'aqi_database.duckdb'
con = duckdb.connect(str(db_path))

# save to duckdb
con.execute("DROP TABLE IF EXISTS aqi_data")
con.execute("CREATE TABLE aqi_data AS SELECT * FROM df")

# run a quick test query
res = con.execute("SELECT City, AVG(\"PM2.5\") as avg_pm25 FROM aqi_data GROUP BY City").df()
print("DuckDB Query successful:\n", res)
con.close()

DuckDB Query successful:
      City   avg_pm25
0  Madrid  24.862959
1  Berlin  16.913505


In [6]:
# save clean parquet
df.to_parquet(processed_dir / 'aqi_clean.parquet')
print("Saved aqi_clean.parquet")

Saved aqi_clean.parquet
